### Install Libraries

In [ ]:
# !!pip install torch transformers datasets peft trl accelerate bitsandbytes

### Import Libraries

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import DPOTrainer, DPOConfig
import torch

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

### Setting peft and Qlora to fit on GPU

In [ ]:
# Optional: 4-bit quantization (QLoRA) to fit on a single GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# LoRA config — inject into attention projection layers
lora_config = LoraConfig(
    r=16,                      # Rank of the decomposition matrices
    lora_alpha=32,             # Scaling factor (usually 2x rank)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"   # MLP layers too for better coverage
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Trainable params: ~20M out of 7B  (≈ 0.28%)

### Load Dataset

In [ ]:
# dataset = load_dataset("trl-internal-testing/hh-rlhf-helpful-base-trl-style", split="train")
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

### Setting DPO config and trainer 

In [ ]:
training_args = DPOConfig(
    output_dir="./dpo-lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,        # Higher LR is fine for LoRA adapters
    beta=0.1,
    max_length=1024,
    # max_prompt_length=512,
    bf16=True,
    optim="paged_adamw_8bit",  # Memory-efficient optimizer for QLoRA
    logging_steps=10,
    save_steps=200,
)

In [ ]:
# With LoRA + PEFT, pass ref_model=None
# TRL automatically uses the base model (pre-adapter) as the reference
trainer = DPOTrainer(
    model=model,
    ref_model=None,            # ← Key difference! TRL infers reference from base weights
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

### To start training run command below

In [ ]:
trainer.train()

### Saved trained model

In [ ]:
# ============================================================
# SAVE
# ============================================================
model.save_pretrained("./dpo-lora-adapters")
tokenizer.save_pretrained("./dpo-lora-adapters")

### Below Step is to Load trained model and predict based on query

In [ ]:
# ============================================================
# LOAD — with optional merge
# ============================================================
sft_path = "./dpo-lora-adapters"
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    sft_path,                   # load the same SFT base that DPO was trained on
    quantization_config=bnb_config,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, "./dpo-lora-adapters")
# Optional merge for faster inference:
# model = model.merge_and_unload()
tokenizer = AutoTokenizer.from_pretrained("./dpo-lora-adapters")


In [ ]:
# ============================================================
# PREDICT
# ============================================================
import torch

def predict(prompt: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(predict("Give me a concise explanation of backpropagation."))